In [1]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from skimage.transform import resize
import random
import os
from PIL import Image
import pandas as pd
import math
from matplotlib.image import imread

print("Done importing libraries\n")

Done importing libraries



### Preprocessing: Linear Contrast Stretching
#### (Applied after the Median and Gaussian filters)

In [2]:
def linear_contrast_stretching(image):
    img = image.copy()
    #Store the min and max intensity in the grayscale dog image.
    min_intensity = np.min(img)  
    max_intensity = np.max(img)  

    linear_contrast_stretched_image = img.copy()
    denominator = max_intensity - min_intensity
    if denominator != 0:  #We can't divide by zero!
        return (img - min_intensity) * (255 / denominator)  #Apply the linear contrast stretching formula.
    else:
        print("Cannot apply linear contrast stretching because all pixel intensities are the same. Please try with a different image.")

print("Defined linear contrast stretching function")

Defined linear contrast stretching function


### Preprocessing: Running the Sobel Operator on Median_Gaussian_Filtered_Images

In [3]:
if not os.path.exists("sobel_median_gaussian_images"):
    os.makedirs("sobel_median_gaussian_images")
    print(f"Created folder: sobel_median_gaussian_images")
else:
    print(f"Folder sobel_median_gaussian_images already exists.")
    
print("Operating Sobel on 10 Images\n")
for i in range(1, 11):
    print("Enhancing Image " + str(i))
    #gaussian filter -> save the result to median_gaussian_filtered_images folder
    current_image = imread("median_gaussian_filtered_images/median_gaussian_img_" + str(i) + ".jpg")
    stretch_result = linear_contrast_stretching(current_image)
    res_img = Image.fromarray(stretch_result)
    res_img.convert("L").save(f"sobel_median_gaussian_images/sobel_median_gaussian_img_{i}.jpg")

Folder sobel_median_gaussian_images already exists.
Operating Sobel on 10 Images

Enhancing Image 1
Enhancing Image 2
Enhancing Image 3
Enhancing Image 4
Enhancing Image 5
Enhancing Image 6
Enhancing Image 7
Enhancing Image 8
Enhancing Image 9
Enhancing Image 10


### Define Otsu Thresholding Function

In [3]:
def otsu(image):
    # First compute the histogram
    histogram, bin_edges = np.histogram(image.flatten(), bins=256, range=(0, 255))
    
    probabilities = histogram / image.size

    # t is the intensity values (0 to 255)
    # ω(t) aka prob_class1
    #prob_class2 = 1 - prob_class1
    cumulative_sum = np.cumsum(probabilities)
    # μ(t) = i * p(i)
    cumulative_mean = np.cumsum(probabilities * np.arange(256))

    #last value of cumulative mean
    global_mean_intensity = cumulative_mean[-1]

    
    denominator = cumulative_sum * (1 - cumulative_sum)

    # Only compute variance where denominator is non-zero. (Can't divide by 0)
    # this is a mask
    valid = denominator > 0
    between_class_variance = np.zeros(256)
    between_class_variance[valid] = (global_mean_intensity * cumulative_sum[valid] - cumulative_mean[valid]) ** 2 / \
                                     denominator[valid]

    threshold = np.argmax(between_class_variance)
    return threshold



### Functions to apply Thresholds to binarize an image

In [2]:
def apply_threshold(image, threshold):
    img = image.copy() 
    one = img >= threshold
    zero = img < threshold
    img[one] = 255
    img[zero] = 0
    return img

In [ ]:
def apply_multiple_thresholds(image, thresholds):
    img = image.copy()